In [ ]:
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from cnpix_local_sleep import atlas as atl
from cnpix_local_sleep import plots
from cnpix_local_sleep.morphological.mua.pipeline.full48h import EXCLUSIVE_STACK, INCLUSIVE_STACK
from cnpix_local_sleep.const import CONTRASTS
from cnpix_local_sleep import off_tables
from pathlib import Path
import pubplots as pp

In [ ]:
plot_for_poster = True

# Publication toggle for the stacked condition-count figures. If True (default):
# x ticks/labels on the bottom row only; no per-axes titles / suptitles; the SPS
# legend is always shown.
plot_for_pub = True

# NREM conditions to show (left column of the stacked figures), user-selectable.
# The Wake column is fixed to the two NOD windows (WAKE_CONDITIONS).
NREM_CONDITIONS = ["Early.REC.NREM", "Late.REC.NREM"]

# OFF source selector. Choose one of:
#   "morphological"          -> per-condition OFFs, method=morphological
#   "morphological-full48h"  -> full-48h OFFs subset by condition, method=morphological,
#                            derived IN MEMORY from the raw whole-recording
#                            detection (no persisted full48h_* parquet)
OFF_SOURCE = "morphological-full48h"
# OFF_CLASS drives the retained (in-memory-only) machinery: contrast diffs and
# scatters. The stacked condition-count figures instead derive all five OFF sets
# (BLAS/CLAS/LLAS + the exclusive partition) from the LLAS superset below.
OFF_CLASS = "llas"

use_in_memory_full48h = OFF_SOURCE == "morphological-full48h"

if OFF_SOURCE in ("morphological", "morphological-full48h"):
    from cnpix_local_sleep.morphological.mua import files
else:
    raise ValueError(f"Unknown OFF_SOURCE: {OFF_SOURCE!r}")

In [ ]:
output_dir = Path(f"./outputs/incline_magnitudes/{OFF_SOURCE}")
output_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
if use_in_memory_full48h:
    # Condition-subset full-48h OFFs, derived in memory from the raw
    # whole-recording detection (no persisted full48h_* parquet). Load the LLAS
    # superset: because BLAS nests inside CLAS inside LLAS, every OFF set used by the stacked
    # condition-count figures is a threshold-mask subset of this frame.
    from cnpix_local_sleep.morphological.pipeline import aggregate_experiment_offs as agg

    offs = agg.load_subset_of_48h_offs("llas")
    smry = agg.summarize_subset_of_48h_offs(OFF_CLASS)
else:
    # Canonical per-condition tables (read from disk; these are the R-facing
    # aggregation outputs). The LLAS file is the superset for OFF-set masking.
    offs = pd.read_parquet(files.get_path("llas_offs.parquet"))
    smry = pd.read_parquet(files.get_path(f"summarized_{OFF_CLASS}_offs.parquet"))

In [ ]:
sorted_structures = atl.sort_structures_by_anterior_posterior(
    offs["structure"].unique()
)
offs["structure"] = pd.Categorical(
    offs["structure"], categories=sorted_structures, ordered=True
)

In [ ]:
# Option A: tab20 palette
structure_palette = dict(
    zip(sorted_structures, sns.color_palette("tab20", n_colors=len(sorted_structures)))
)

# Option B: Official BrainGlobeAtlas colors
# atlas = atl.get_atlas()
# atlas_colors = {
# s["acronym"]: (
# s["rgb_triplet"][0] / 255,
# s["rgb_triplet"][1] / 255,
# s["rgb_triplet"][2] / 255,
# )
# for s in atlas.structures_list
# }
# structure_palette = {s: atlas_colors[s] for s in sorted_structures}

In [ ]:
WAKE_CONDITIONS = ["Early.NOD.Wake", "Late.NOD.Wake"]

DIFF_METRICS = [
    "median_duration",
    "mean_boxcox_median_duration",
    "median_median_duration",
    "mean_boxcox_span",
    "median_span",
    "mean_boxcox_area",
    "median_area",
    "rate",
    "total_area_norm",
]

# Per-structure metadata columns that must be removed before differencing
# metrics across conditions: clade/Cx.AP.group are strings (can't subtract)
# and AP.Coord is a per-structure constant (would difference to zeros).
SMRY_DROP_COLS = [
    "clade",
    "AP.Coord",
    "Cx.AP.group",
]


def add_sps_column(df, poster_mode=False):
    """Add ordered subject_probe_structure categorical column.

    If poster_mode is True, anonymize subjects as "Subject1", "Subject2",
    etc. and omit the probe, producing labels like "Subject2, MO".
    """
    df = df.copy()
    # ``subject`` arrives as a pandas Categorical; cast to str before mapping
    # so string concatenation works (a categorical .map can stay categorical,
    # which can't be added to a str).
    subject_str = df["subject"].astype(str)
    if poster_mode:
        subject_map = {
            s: f"Subject{i}"
            for i, s in enumerate(sorted(df["subject"].unique()), start=1)
        }
        df["sps"] = subject_str.map(subject_map) + ", " + df["structure"].astype(str)
    else:
        df["sps"] = (
            subject_str
            + "_"
            + df["probe"].astype(str)
            + "_"
            + df["structure"].astype(str)
        )
    structure_order = {s: i for i, s in enumerate(sorted_structures)}
    sep = ", " if poster_mode else "_"
    sorted_sps = sorted(
        df["sps"].unique(),
        key=lambda sps: structure_order.get(sps.split(sep)[-1], len(sorted_structures)),
    )
    df["sps"] = pd.Categorical(df["sps"], categories=sorted_sps, ordered=True)
    return df


def compute_diffs(smry, normalization="absolute"):
    """Compute contrast diffs from summary data.

    normalization: "absolute", "percent", or "bsl_normalized"
    """
    _smry = smry.drop(columns=SMRY_DROP_COLS)
    _smry = _smry.set_index(["subject", "probe", "structure", "condition"])
    diff_dfs = {}
    for diff, (cnd1, cnd2) in CONTRASTS.items():
        base = _smry.xs(cnd2, level="condition")
        change = _smry.xs(cnd1, level="condition") - base
        if normalization == "percent":
            change = (change / base.abs()) * 100
        elif normalization == "bsl_normalized":
            bsl = _smry.xs("Early.REC.NREM.Match", level="condition").abs()
            change = change / bsl
        diff_dfs[diff] = change.add_suffix(f"_{diff}")
    diffs = pd.concat(diff_dfs.values(), axis=1).reset_index()
    return diffs


def plot_diff_bars(diffs, col, by="sps"):
    """Bar plot of a diff metric, ordered by magnitude.

    by: "sps" for one bar per subject_probe_structure,
        "structure" to aggregate across subjects/probes.
    """
    fig, ax = plt.subplots(figsize=(16, 6))

    if by == "sps":
        plot_data = diffs[["sps", "structure", col]].copy()
        plot_data = plot_data.sort_values(col, ascending=False).dropna()
        sns.barplot(
            plot_data,
            x="sps",
            y=col,
            hue="structure",
            hue_order=sorted_structures,
            palette=structure_palette,
            order=plot_data["sps"],
            ax=ax,
            dodge=False,
        )
        ax.set_xlabel("SPS")
        ax.legend(
            bbox_to_anchor=(1.05, 1),
            loc="upper left",
            title="Structure",
        )
    else:
        plot_data = diffs[["structure", col]].dropna()
        order = (
            plot_data.groupby("structure")[col]
            .median()
            .sort_values(ascending=False)
            .index.tolist()
        )
        sns.barplot(
            plot_data,
            x="structure",
            y=col,
            order=order,
            palette=structure_palette,
            ax=ax,
        )
        ax.set_xlabel("Structure")

    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
    ax.set_ylabel(col)
    ax.set_title(f"{col} (ordered by magnitude)")
    plt.tight_layout()
    plt.show()
    return fig, ax


def plot_wake_nrem_barplots(counts, y_col, hue_col, palette):
    """Split by wake/NREM and plot side-by-side barplots."""
    counts = counts.copy()
    counts["condition_type"] = counts["condition"].apply(
        lambda x: "Wake" if x in WAKE_CONDITIONS else "NREM"
    )
    fig, axes = plt.subplots(
        1,
        2,
        figsize=pp.scale(12, 4),
        gridspec_kw={"width_ratios": [1, 2]},
    )
    for ax, ctype in zip(axes, ["Wake", "NREM"]):
        subset = counts[counts["condition_type"] == ctype].copy()
        subset["condition"] = subset["condition"].cat.remove_unused_categories()
        order = (
            subset.groupby(hue_col)[y_col]
            .mean()
            .sort_values(ascending=False)
            .index.tolist()
        )
        sns.barplot(
            data=subset,
            x="condition",
            y=y_col,
            hue=hue_col,
            hue_order=order,
            palette=palette,
            ax=ax,
            dodge=True,
            alpha=0.7,
        )
        ax.set_title(f"{ctype} Conditions")
        ax.tick_params(axis="x", rotation=45)
        ax.set_xlabel("Condition")
        ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
        if ctype == "NREM" and "frac_bsl" in y_col:
            ax.axhline(1.0, color="black", linestyle="--", linewidth=0.8)
    plt.tight_layout()
    plt.show()
    return fig, axes

In [ ]:
# --- Stacked condition-count figures (publication) ------------------------
# One row per OFF set; Wake conditions (left) and NREM conditions (right). Built
# with plots.stacked_rows so the rows line up with the intrusion-sweep figures
# when placed side by side (same nrows/height/top/bottom/hspace, ~3.3" tall).

FILTER_LABELS = {
    "blas": "BLAS",
    "clas": "CLAS",
    "llas": "LLAS",
    "clas-exclusive": "CLAS-exclusive",
    "llas-exclusive": "LLAS-exclusive",
}


def build_off_set_masks(offs):
    """Boolean masks (aligned to ``offs.index``) for the five OFF sets.

    ``offs`` must be the LLAS superset. BLAS nests inside CLAS inside LLAS, so the exclusive
    categories are adjacent mask differences and BLAS + CLAS-exclusive +
    LLAS-exclusive reconstructs LLAS.
    """
    mask_blas = off_tables.off_filter_mask(offs, "blas")
    mask_clas = off_tables.off_filter_mask(offs, "clas")
    all_true = pd.Series(True, index=offs.index)
    return {
        "blas": mask_blas,
        "clas": mask_clas,
        "llas": all_true,
        "clas-exclusive": mask_clas & ~mask_blas,
        "llas-exclusive": ~mask_clas,  # within LLAS: llas & ~clas
    }


def sps_order_by_count(offs, mask, condition, all_sps):
    """SPS ordered by descending OFF count in ``condition`` on the masked set.

    Reindexed to the full ``all_sps`` list (missing -> 0) so the ordering, and
    thus the bar order, is identical across every OFF-set row.
    """
    counts = (
        offs.loc[mask]
        .groupby(["sps", "condition"], observed=True)
        .size()
        .reset_index(name="count")
    )
    s = (
        counts[counts["condition"] == condition]
        .set_index("sps")["count"]
        .reindex(all_sps)
        .fillna(0)
    )
    return s.sort_values(ascending=False).index.tolist()


def plot_condition_counts_stack(
    offs,
    set_masks,
    filter_order,
    *,
    wake_conditions,
    nrem_conditions,
    wake_order,
    nrem_order,
    sps_palette,
    structure_palette,
    out_path,
    plot_for_pub=True,
    destination="figma",
    width_in=3.1,
):
    """3-row (OFF set) x 2-col (Wake left, NREM right) OFF-count bar figure.

    Bars are subject-probe-structure (SPS) level, colored by structure. Their
    within-condition order is fixed by ``wake_order`` / ``nrem_order`` (SPS
    orders derived once from the CLAS set), identical across all rows. The
    legend is condensed to one entry per structure.
    """
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    cols = {"Wake": list(wake_conditions), "NREM": list(nrem_conditions)}
    orders = {"Wake": wake_order, "NREM": nrem_order}
    col_order = ["Wake", "NREM"]
    n = len(filter_order)
    width_ratios = [len(cols["Wake"]), len(cols["NREM"])]

    with pp.destination(destination):
        fig, axes = plots.stacked_rows(
            width_in,
            ncols=2,
            nrows=n,
            width_ratios=width_ratios,
            left=0.16,
            right=0.72,
            wspace=0.4,
        )
        for i, filter_name in enumerate(filter_order):
            sub = offs.loc[set_masks[filter_name]]
            counts = (
                sub.groupby(["sps", "condition"], observed=True)
                .size()
                .reset_index(name="count")
            )
            is_bottom = i == n - 1
            for j, ctype in enumerate(col_order):
                ax = axes[i, j]
                conds = cols[ctype]
                cdata = counts[counts["condition"].isin(conds)].copy()
                cdata["condition"] = pd.Categorical(
                    cdata["condition"], categories=conds, ordered=True
                )
                sns.barplot(
                    data=cdata,
                    x="condition",
                    y="count",
                    hue="sps",
                    hue_order=orders[ctype],
                    palette=sps_palette,
                    ax=ax,
                    dodge=True,
                    legend=False,
                )
                ax.set_xlabel("")
                ax.set_ylabel("OFF count" if j == 0 else "")
                if is_bottom:
                    ax.tick_params(axis="x", rotation=45)
                    for lbl in ax.get_xticklabels():
                        lbl.set_ha("right")
                elif plot_for_pub:
                    ax.tick_params(bottom=False, labelbottom=False)
                if not plot_for_pub:
                    ax.set_title(f"{FILTER_LABELS[filter_name]}: {ctype}")
        # Legend condensed to structure (bars are SPS-level, colored by
        # structure; individual SPS are not resolvable to the reader).
        handles = [
            mpatches.Patch(color=c, label=s) for s, c in structure_palette.items()
        ]
        fig.legend(
            handles=handles,
            loc="center left",
            bbox_to_anchor=(0.73, 0.5),
            fontsize=pp.scale(5),
            frameon=False,
            labelspacing=0.2,
            handlelength=pp.scale(0.9),
            title="Structure",
            title_fontsize=pp.scale(5),
        )
        fig.savefig(out_path)
        plt.show()
    return fig, axes

In [ ]:
offs = add_sps_column(offs, poster_mode=plot_for_poster)

if plot_for_poster:
    sps_palette = {
        sps: structure_palette[sps.split(", ")[-1]]
        for sps in offs["sps"].cat.categories
    }
else:
    sps_palette = {
        sps: structure_palette[sps.rsplit("_", 1)[-1]]
        for sps in offs["sps"].cat.categories
    }

In [ ]:
for i, k in enumerate(sps_palette):
    print(f"{i}: {k}")

In [ ]:
# Mapping from real subject name -> anonymized label, using the same
# scheme as add_sps_column (subjects sorted, numbered from 1). E.g.
# "CNPIX12-Santiago" -> "Subject3".
subject_anon_map = {
    s: f"Subject{i}"
    for i, s in enumerate(sorted(offs["subject"].astype(str).unique()), start=1)
}
subject_anon_map

## OFF counts by condition

The publication deliverable: two 3-row stacked figures, one row per OFF set, with Wake
conditions on the left and NREM on the right. Bars are subject-probe-structure (SPS)
level, colored by structure, and the legend is condensed to structure. Within each
column the SPS bar order is fixed from the CLAS set and reused on every row: Wake by
CLAS `Late.NOD.Wake` count, NREM by CLAS `Late.REC.NREM` count. These are the only
figures written to disk. `inclusive.svg` stacks BLAS/CLAS/LLAS; `exclusive.svg` stacks
BLAS / CLAS-exclusive / LLAS-exclusive, the adjacent partition. Built ~3.1 inches wide by
3.3 tall to line up row-for-row with the `intrusion_sweep` figures.


In [ ]:
# Five OFF sets from the LLAS superset, and the two stacked deliverable figures.
set_masks = build_off_set_masks(offs)

# Partition sanity check: BLAS + CLAS-exclusive + LLAS-exclusive == LLAS.
_partition = (
    set_masks["blas"] | set_masks["clas-exclusive"] | set_masks["llas-exclusive"]
)
assert bool(_partition.all()), "exclusive OFF sets do not partition LLAS"

# Bar orders (SPS, descending), fixed once from the CLAS set and reused on every
# row: Wake by CLAS Late.NOD.Wake count, NREM by CLAS Late.REC.NREM count.
all_sps = list(sps_palette.keys())
wake_order = sps_order_by_count(offs, set_masks["clas"], "Late.NOD.Wake", all_sps)
nrem_order = sps_order_by_count(offs, set_masks["clas"], "Late.REC.NREM", all_sps)

_stack_kwargs = dict(
    wake_conditions=WAKE_CONDITIONS,
    nrem_conditions=NREM_CONDITIONS,
    wake_order=wake_order,
    nrem_order=nrem_order,
    sps_palette=sps_palette,
    structure_palette=structure_palette,
    plot_for_pub=plot_for_pub,
)

fig_incl, _ = plot_condition_counts_stack(
    offs,
    set_masks,
    INCLUSIVE_STACK,
    out_path=output_dir / "condition_off_counts_inclusive.svg",
    **_stack_kwargs,
)
fig_excl, _ = plot_condition_counts_stack(
    offs,
    set_masks,
    EXCLUSIVE_STACK,
    out_path=output_dir / "condition_off_counts_exclusive.svg",
    **_stack_kwargs,
)

In [ ]:
# Machinery (kept in memory, not written to disk): raw counts by structure,
# on the LLAS set. Rendered inline for inspection only.
counts_struct = (
    offs.groupby(["subject", "probe", "structure", "condition"], observed=True)
    .size()
    .reset_index(name="count")
).copy()
with pp.destination("figma"):
    plot_wake_nrem_barplots(counts_struct, "count", "structure", structure_palette)

In [ ]:
# Machinery (in memory, not saved): counts as fraction of BSL NREM, by sps.
counts_sps = (
    offs.groupby(["sps", "condition"], observed=True).size().reset_index(name="count")
)
bsl_counts = counts_sps[counts_sps["condition"] == "Early.REC.NREM.Match"][
    ["sps", "count"]
].rename(columns={"count": "bsl_count"})
counts_sps_frac = counts_sps.merge(bsl_counts, on="sps", how="left")
counts_sps_frac["count_frac_bsl"] = (
    counts_sps_frac["count"] / counts_sps_frac["bsl_count"]
)
with pp.destination("figma"):
    plot_wake_nrem_barplots(counts_sps_frac, "count_frac_bsl", "sps", sps_palette)

In [ ]:
# Machinery (in memory, not saved): counts as fraction of BSL NREM, by structure.
bsl_struct_counts = counts_struct[counts_struct["condition"] == "Early.REC.NREM.Match"][
    ["subject", "probe", "structure", "count"]
].rename(columns={"count": "bsl_count"})
counts_struct_frac = counts_struct.merge(
    bsl_struct_counts,
    on=["subject", "probe", "structure"],
    how="left",
)
counts_struct_frac["count_frac_bsl"] = (
    counts_struct_frac["count"] / counts_struct_frac["bsl_count"]
)
with pp.destination("figma"):
    plot_wake_nrem_barplots(
        counts_struct_frac,
        "count_frac_bsl",
        "structure",
        structure_palette,
    )

In [ ]:
smry.columns

In [ ]:
# Sanity checks. detection_mode/layer/threshold_group are no longer emitted
# by the aggregation, so only assert on columns that are still present.
assert all(smry["clade"] == "Cx")
for _col, _val in [
    ("detection_mode", "spatial"),
    ("layer", "None"),
    ("threshold_group", "None"),
]:
    if _col in smry.columns:
        assert all(smry[_col] == _val)

## Contrast diff bar plots

In [ ]:
# Machinery (in memory, not saved): contrast diff bar plots by sps.
for norm_mode, title in [
    ("percent", "Percent change"),
    ("absolute", "Absolute, unnormalized change"),
    ("bsl_normalized", "Change, normalized by Early BSL NREM"),
]:
    diffs = compute_diffs(smry, normalization=norm_mode)
    diffs = add_sps_column(diffs, poster_mode=plot_for_poster)
    print(f"\n{'=' * 60}")
    print(title)
    print(f"{'=' * 60}")
    for metric in DIFF_METRICS:
        with pp.destination("figma"):
            plot_diff_bars(diffs, f"{metric}_NOD.Incline")

In [ ]:
# Machinery (in memory, not saved): diff bar plots by structure.
for norm_mode, title in [
    ("percent", "Percent change"),
    ("absolute", "Absolute, unnormalized change"),
    ("bsl_normalized", "Change, normalized by Early BSL NREM"),
]:
    diffs = compute_diffs(smry, normalization=norm_mode)
    diffs = add_sps_column(diffs, poster_mode=plot_for_poster)
    print(f"\n{'=' * 60}")
    print(title)
    print(f"{'=' * 60}")
    for metric in DIFF_METRICS:
        with pp.destination("figma"):
            plot_diff_bars(diffs, f"{metric}_NOD.Incline", by="structure")

In [ ]:
# Machinery (in memory, not saved): scatter of duration vs span, sized by area.
diffs_bsl = compute_diffs(smry, normalization="bsl_normalized")
diffs_bsl = add_sps_column(diffs_bsl, poster_mode=plot_for_poster)

with pp.destination("figma"):
    fig, ax = plt.subplots(figsize=pp.scale(4, 3))
    sns.scatterplot(
        diffs_bsl,
        x="mean_boxcox_median_duration_NOD.Incline",
        y="mean_boxcox_span_NOD.Incline",
        size="total_area_norm_NOD.Incline",
        ax=ax,
    )

In [ ]:
diffs_bsl = compute_diffs(smry, normalization="bsl_normalized")
diffs_bsl = add_sps_column(diffs_bsl, poster_mode=plot_for_poster)

with pp.destination("figma"):
    fig, ax = plt.subplots(figsize=pp.scale(4, 3))
    sns.scatterplot(
        diffs_bsl,
        x="total_area_norm_NREM.Rebound",
        y="total_area_norm_NOD.Incline",
        ax=ax,
    )

In [ ]:
diffs_bsl = compute_diffs(smry, normalization="bsl_normalized")
diffs_bsl = add_sps_column(diffs_bsl, poster_mode=plot_for_poster)

with pp.destination("figma"):
    fig, ax = plt.subplots(figsize=pp.scale(4, 3))
    sns.scatterplot(
        diffs_bsl,
        x="count_NREM.Rebound",
        y="count_NOD.Incline",
        ax=ax,
    )

In [ ]:
diffs_bsl = compute_diffs(smry, normalization="bsl_normalized")
diffs_bsl = add_sps_column(diffs_bsl, poster_mode=plot_for_poster)

with pp.destination("figma"):
    fig, ax = plt.subplots(figsize=pp.scale(4, 3))
    sns.scatterplot(
        diffs_bsl,
        x="rate_NREM.Rebound",
        y="rate_NOD.Incline",
        ax=ax,
    )

In [ ]:
diffs_bsl = compute_diffs(smry, normalization="absolute")
diffs_bsl = add_sps_column(diffs_bsl, poster_mode=plot_for_poster)

with pp.destination("figma"):
    fig, ax = plt.subplots(figsize=pp.scale(4, 3))
    sns.scatterplot(
        diffs_bsl,
        x="total_area_norm_NREM.Rebound",
        y="total_area_norm_NOD.Incline",
        ax=ax,
    )

In [ ]:
diffs_bsl = compute_diffs(smry, normalization="absolute")
diffs_bsl = add_sps_column(diffs_bsl, poster_mode=plot_for_poster)

with pp.destination("figma"):
    fig, ax = plt.subplots(figsize=pp.scale(4, 3))
    sns.scatterplot(
        diffs_bsl,
        x="count_NREM.Rebound",
        y="count_NOD.Incline",
        ax=ax,
    )

In [ ]:
diffs_bsl = compute_diffs(smry, normalization="absolute")
diffs_bsl = add_sps_column(diffs_bsl, poster_mode=plot_for_poster)

with pp.destination("figma"):
    fig, ax = plt.subplots(figsize=pp.scale(4, 3))
    sns.scatterplot(
        diffs_bsl,
        x="rate_NREM.Rebound",
        y="rate_NOD.Incline",
        ax=ax,
    )